# 社交图好友推荐与链接预测实验

## tl;dr

在SNAP Facebook图的5个连通性保护划分上，Resource Allocation取得最佳NDCG@10=0.5283，相对Common Neighbors提高3.56个百分点；Recall@10提高3.75个百分点。

## Context & Methods

目标是把共同好友排序升级为可离线评价的链接预测实验。每个随机种子保护完整图的一棵生成树，再隐藏10%的非树边，确保训练图连通且测试边不泄漏。主AUC使用共享至少一个训练邻居的困难负边。

### Key Assumptions

- SNAP合并图是静态无向图，没有时间戳，因此结果反映结构恢复能力，不代表线上因果收益。
- Top-K候选来自训练图二跳邻居，符合朋友的朋友召回场景。


## Data


In [1]:
import csv, json
from pathlib import Path
root = Path('..').resolve()
metadata = json.loads((root / 'results' / 'metadata.json').read_text(encoding='utf-8'))
print(metadata)


{'nodes': 4039, 'edges': 88234, 'seeds': [11, 22, 33, 44, 55], 'test_fraction': 0.1, 'k': 10, 'source': 'Stanford SNAP ego-Facebook combined graph', 'source_url': 'https://snap.stanford.edu/data/ego-Facebook.html'}


## Results


In [2]:
with (root / 'results' / 'summary.csv').open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
for row in rows:
    print(row['method'], 'hard_auc=', round(float(row['hard_auc_mean']), 4), 'recall@10=', round(float(row['recall_at_k_mean']), 4), 'ndcg@10=', round(float(row['ndcg_at_k_mean']), 4))


resource_allocation hard_auc= 0.9772 recall@10= 0.5617 ndcg@10= 0.5283
adamic_adar hard_auc= 0.9705 recall@10= 0.5431 ndcg@10= 0.5102
common_neighbors hard_auc= 0.9657 recall@10= 0.5242 ndcg@10= 0.4927
jaccard hard_auc= 0.9655 recall@10= 0.5072 ndcg@10= 0.4807


![NDCG comparison](../results/ndcg_at_10.svg)

![Hard-negative AUC comparison](../results/auc.svg)


In [3]:
with (root / 'results' / 'shuffle_results.csv').open(encoding='utf-8') as handle:
    shuffle = list(csv.DictReader(handle))
mean_reduction = sum(float(row['record_reduction']) for row in shuffle) / len(shuffle)
print('mean combiner record reduction:', f'{mean_reduction:.2%}')


mean combiner record reduction: 70.73%


## Takeaways

1. 对高聚类社交图，降低高连接公共邻居权重的Resource Allocation优于原始共同好友计数。
2. 随机负边使AUC接近1，主报告应使用二跳困难负边和Top-K排序指标。
3. 本地聚合显著减少中间记录，但这里只模拟记录量；原四节点Spark/Hadoop运行时间需在原集群环境复测。
